# CatVTON prototype on Kaggle
Enable Internet and a T4 accelerator. Use only consented images. Non-commercial research/testing only; a free app is not automatically non-commercial. This notebook has not been GPU-executed by Codex.

Creates an isolated Python 3.11 environment so Kaggle’s system packages remain intact. One T4 is used; two T4s do not combine their memory. Setup downloads several GB.

In [ ]:
import os, subprocess, sys
from pathlib import Path
ROOT = Path('/kaggle/working/fitsyncgemini')
# After merging the PR use main; before merging use the PR branch below.
BRANCH = 'codex/async-vton-prototype'
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/sadad54/fitsyncgemini.git',str(ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)


In [ ]:
VENV = Path('/kaggle/working/catvton-env')
subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(VENV)],check=True)
PYTHON = str(VENV/'bin/python')
subprocess.run([sys.executable,'-m','uv','pip','install','--python',PYTHON,'-r',str(ROOT/'gpu_worker/requirements.txt')],check=True)
CAT = ROOT/'gpu_worker/CatVTON'
if not CAT.exists(): subprocess.run(['git','clone','https://github.com/Zheng-Chong/CatVTON.git',str(CAT)],check=True)
subprocess.run(['git','-C',str(CAT),'checkout','999bdbe81e6008a3f5749af7c1e0b0fa3d21b48e'],check=True)
os.environ['CATVTON_DIR'] = str(CAT)
subprocess.run([PYTHON,'-c','import torch; print(torch.cuda.get_device_name(0)); assert torch.cuda.is_available()'],check=True)


## Compare on your own test set
Copy the example manifest, point it at images attached as a private Kaggle dataset, and set MANIFEST below. Include at least 10 dresses (short/long, fitted/loose, patterned/plain), plus tops/bottoms. Preprocessing writes masks and coloured DensePose inputs for both models. Inspect those masks before accepting results. These scripts store photos in notebook outputs: keep outputs private.

In [ ]:
MANIFEST = Path('/kaggle/input/your-private-dataset/manifest.json')  # Change this
PREPARED = Path('/kaggle/working/vton-prepared')
RESULTS = Path('/kaggle/working/vton-results')
subprocess.run([PYTHON,str(ROOT/'gpu_worker/benchmark.py'),'prepare',str(MANIFEST),str(PREPARED)],check=True)
subprocess.run([PYTHON,str(ROOT/'gpu_worker/benchmark.py'),'catvton',str(PREPARED/'prepared.json'),str(RESULTS)],check=True)


## Optional supervised app test
Only if Kaggle currently permits your temporary tunnel use. Kaggle is not production hosting. Add GPU_TRYON_SHARED_SECRET (at least 32 random characters) and NGROK_AUTHTOKEN through Kaggle Secrets; do not paste them into cells. The ngrok URL changes each session. Stop the tunnel when finished. No Supabase key belongs in this notebook.

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['GPU_TRYON_SHARED_SECRET'] = secrets.get_secret('GPU_TRYON_SHARED_SECRET')
os.environ['TRYON_NONCOMMERCIAL_ACK'] = 'true'
# Use the notebook Python only for the tunnel; model server uses its own venv.
subprocess.run([sys.executable,'-m','pip','install','pyngrok==7.2.12'],check=True)
from pyngrok import ngrok
ngrok.set_auth_token(secrets.get_secret('NGROK_AUTHTOKEN'))
log = open('/kaggle/working/worker.log','w')
server = subprocess.Popen([PYTHON,'-m','uvicorn','app:app','--host','0.0.0.0','--port','8100'],cwd=ROOT/'gpu_worker',env=os.environ.copy(),stdout=log,stderr=log)
tunnel = ngrok.connect(8100,bind_tls=True)
print('Set TRYON_ENDPOINT in backend/.env to:',tunnel.public_url)


In [ ]:
# Run when finished. This ends the temporary app test.
ngrok.disconnect(tunnel.public_url)
server.terminate()
log.close()
